# Stage 5 §H Phase A — TSP-20 mixed leaf-eval λ-sweep (Colab T4)

Inference-only λ-sweep on the F.6.1.6 step-decay checkpoint (the most-trained TSP-20 value head we have). For each λ ∈ {0.0, 0.25, 0.5, 0.75, 1.0} runs MCTS with `leaf_eval='mix', mix_lambda=λ` on `val_size=10000, val_seed=42, K=40, ε=0, τ=0`.

Reference anchors (§C.3 / §D.5 on F.6.1.6 at 2000 instances): λ=0 ≈ 3.834 (pure rollout); λ=1 ≈ 3.868 (pure value head). An interior λ winning would clear the §H hypothesis.

**Before running:** upload `outputs/tsp_20/f616_400iter_step_decay_20260507T101222_20260507T101229/iter-361_accepted.pt` from your local repo to Drive at

```
MyDrive/AM_AlphaGoZero/checkpoints/f616_400iter_step_decay/iter-361_accepted.pt
```

**Wall expectation on T4:** ~10–15 min per λ (one value-head MLP call per leaf on top of the rollout decode budget), so the full 5-λ sweep is ~60–90 min. CSV + per-instance NPZ get written to Drive for persistence.

## Section 1 — setup (Drive mount + repo + build)

### 1.1 GPU / Python / CUDA sanity

In [ ]:
import sys, platform
print('python    =', sys.version.split()[0], platform.platform())
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch     =', torch.__version__, '  cuda available =', torch.cuda.is_available())
print('cuda      =', torch.version.cuda, '  device =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

### 1.2 Mount Drive + workspace paths

Mirrors the layout used by `colab_setup_and_validate.ipynb`. `WORKSPACE/checkpoints/f616_400iter_step_decay/iter-361_accepted.pt` is where the F.6.1.6 checkpoint must live before the sweep runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKSPACE = '/content/drive/MyDrive/AM_AlphaGoZero'
REPO_DIR = os.path.join(WORKSPACE, 'repo')
CKPT_DIR = os.path.join(WORKSPACE, 'checkpoints')
OUTPUT_DIR = os.path.join(WORKSPACE, 'outputs')

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('WORKSPACE =', WORKSPACE)
print('REPO_DIR  =', REPO_DIR)
print('CKPT_DIR  =', CKPT_DIR)
print('OUTPUT_DIR=', OUTPUT_DIR)

### 1.3 Clone or update repo

In [ ]:
REPO_URL = 'https://github.com/LejunZhou/AM_ALPHAGOZERO.git'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned; pulling latest...')
    !git -C {REPO_DIR} pull --ff-only

!git -C {REPO_DIR} log --oneline -1

### 1.4 Install package + build C++ MCTS extension

`--no-deps` keeps Colab's pre-installed CUDA torch. The C++ extension is plain pybind11 (no torch C++ ABI), so it builds against whatever g++ Colab ships.

In [ ]:
%cd {REPO_DIR}
!pip install --quiet pybind11
!pip install --quiet --no-deps -e .
!pip install --quiet matplotlib numpy scipy tqdm 'wandb>=0.18.0'

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
from am_baseline.search.mcts_cpp import _mcts_cpp as _ext  # noqa: F401
from am_baseline.search.mcts import MCTSConfig, MCTSSolver
assert 'mix' in MCTSSolver.VALID_LEAF_EVAL, 'mix mode missing — pull latest commits and rebuild'
print('OK — C++ MCTS extension imports; mix mode registered')
print('VALID_LEAF_EVAL =', MCTSSolver.VALID_LEAF_EVAL)

### 1.5 Smoke test — confirm mix mode is wired correctly

Runs the three parity checks from `src/scripts/smoke_mix.py` (M1: rollout↔mix(λ=0), M2: value_head↔mix(λ=1), M3: Python↔C++ at mix(λ=0.5)). All three must report `|Δ|=0` before the sweep.

In [ ]:
!cd {REPO_DIR} && PYTHONPATH=src python src/scripts/smoke_mix.py

## Section 2 — Phase A λ-sweep on F.6.1.6

### 2.1 Verify F.6.1.6 checkpoint is on Drive

In [ ]:
CKPT = os.path.join(CKPT_DIR, 'f616_400iter_step_decay', 'iter-361_accepted.pt')
if not os.path.isfile(CKPT):
    raise FileNotFoundError(
        f'Missing checkpoint at {CKPT}.\n'
        f'Upload outputs/tsp_20/f616_400iter_step_decay_20260507T101222_20260507T101229/iter-361_accepted.pt '
        f'from your local repo to that Drive path before running the sweep.'
    )
print('Checkpoint found:', CKPT, '(', os.path.getsize(CKPT) // 1024, 'KB)')

### 2.2 Run the λ-sweep

Full grid: λ ∈ {0.0, 0.25, 0.5, 0.75, 1.0}. CSV + per-instance NPZ land in Drive at `OUTPUT_DIR/eval_logs/`.

In [ ]:
OUT_CSV = os.path.join(OUTPUT_DIR, 'eval_logs', 'tsp20_f616_mix_lambda_sweep_K40.csv')
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

!cd {REPO_DIR} && PYTHONPATH=src python src/scripts/eval_tsp20_mix_lambda_sweep.py \
    --ckpt {CKPT} \
    --K 40 \
    --val_size 10000 \
    --val_seed 42 \
    --lambdas 0.0,0.25,0.5,0.75,1.0 \
    --device cuda \
    --mcts_batch_size 1000 \
    --out_csv {OUT_CSV}

### 2.3 Results table + endpoint anchor check

Loads the CSV, prints a tidy table, and flags whether the λ=0 / λ=1 endpoints match the §C.3 / §D.5 anchors from F.6.1.6. Per-instance NPZ kept on Drive for paired-t analysis downstream.

In [ ]:
import csv
import numpy as np

with open(OUT_CSV, newline='') as f:
    rows = list(csv.DictReader(f))

print(f"\n=== Phase A: TSP-20 mix-λ sweep on F.6.1.6 (K={rows[0]['K']}, val_size={rows[0]['val_size']}, seed={rows[0]['val_seed']}) ===")
print(f"{'λ':>6} {'mean':>10} {'SE':>9} {'wall (s)':>10} {'decode':>12} {'value':>10} {'rollout':>12}")
print('-' * 80)
for r in rows:
    lam = float(r['mix_lambda'])
    print(f"{lam:>6.2f} {float(r['mean_cost']):>10.5f} {float(r['se_cost']):>9.5f} {float(r['wall_s']):>10.1f} {int(r['fwd_decode']):>12d} {int(r['fwd_value']):>10d} {int(r['fwd_rollout']):>12d}")
print('-' * 80)

# Anchor check vs §C.3 / §D.5 numbers on F.6.1.6 (2000-instance probes).
ROLLOUT_ANCHOR = 3.834
VH_ANCHOR = 3.868
ANCHOR_TOL = 0.01  # 10000-instance SE on F.6.1.6 is ~0.004; allow 2.5x for cross-GPU FP

lambdas = np.array([float(r['mix_lambda']) for r in rows])
means = np.array([float(r['mean_cost']) for r in rows])
for lam_target, anchor, label in [(0.0, ROLLOUT_ANCHOR, 'pure rollout'),
                                    (1.0, VH_ANCHOR, 'pure value_head')]:
    if lam_target in lambdas:
        idx = int(np.where(lambdas == lam_target)[0][0])
        delta = means[idx] - anchor
        flag = 'OK' if abs(delta) < ANCHOR_TOL else 'CHECK'
        print(f'  [{flag}] λ={lam_target}  ({label}):  measured {means[idx]:.5f}  vs anchor {anchor:.3f}  Δ = {delta:+.5f}')

# Identify the winning λ (lowest mean cost).
winner_idx = int(np.argmin(means))
winner_lambda = float(lambdas[winner_idx])
winner_mean = float(means[winner_idx])
interior = winner_lambda not in (0.0, 1.0)
print()
print(f'  Lowest mean cost: λ={winner_lambda}  mean={winner_mean:.5f}  '
      f"({'INTERIOR λ wins — mix is helping' if interior else 'endpoint — mix offers no inference benefit on this checkpoint'})")
print(f'\nCSV: {OUT_CSV}')
print(f'NPZ: {OUT_CSV.replace(".csv", ".npz")}  (per-instance costs for paired-t analysis)')

### 2.4 Next steps

- If the table shows an interior λ wins (especially with statistical significance vs λ=0): use that λ (or the top 1–2) for Phase B Modal training (`run_tsp20_k10_mix_step50`).
- If the curve is monotone (λ=0 strictly wins): F.6.1.6's biased value head can't be salvaged by inference mixing. Phase B still informative because from-scratch mix self-play may train a less-biased value head.
- Either way, copy the CSV and the printed table into `_progress/stage5_mix_leafeval_progress.md` §H.3 and commit.